Agent


In [1]:
from abc import ABC, abstractmethod
import random
import numpy as np

class Agent(ABC):
    """
    Abstract Base Class for all market participants.
    Enforces that every agent must have a 'get_action' method.
    """
    def __init__(self, agent_id, initial_cash, initial_inventory):
        self.agent_id = agent_id
        self.cash = initial_cash
        self.inventory = initial_inventory
        
    @abstractmethod
    def get_action(self, market_snapshot, fair_value=None):
        """
        Input: 
            market_snapshot (dict): Contains 'best_bid', 'best_ask', 'mid_price', 'time'
            fair_value (float): Optional 'Fundamental Value' for agents who know it.
        
        Output: 
            Order dictionary or None
            Format: {'side': 'buy'|'sell', 'price': float, 'qty': int, 'type': 'limit'|'market'}
        """
        pass

class RandomAgent(Agent):
    """
    Day 6 Agent: A 'Zero Intelligence' agent for testing.
    It buys or sells randomly, ignoring external signals.
    """
    def __init__(self, agent_id, cash, inventory, arrival_rate=1.0):
        super().__init__(agent_id, cash, inventory)
        self.arrival_rate = arrival_rate 

    def get_action(self, market_snapshot, fair_value=None):
        # 1. Poisson Arrival Check 
        if random.random() > self.arrival_rate:
            return None 

        side = 'buy' if random.random() > 0.5 else 'sell'
        
        # Determine Reference Price (Mid Price or 100 fallback)
        mid_price = market_snapshot.get('mid_price')
        ref_price = mid_price if mid_price else 100.0
        
        if side == 'buy':
            price = round(ref_price + random.uniform(0, 2), 2)
        else:
            price = round(ref_price - random.uniform(0, 2), 2)
            
        qty = random.randint(1, 10)
        
        return {
            'agent_id': self.agent_id,
            'side': side,
            'price': price,
            'qty': qty,
            'type': 'limit'
        }

class NoiseTrader(Agent):
    """
    Day 7 Agent: Liquidity Consumer.
    Follows a private 'Fair Value' (Brownian Motion) and places orders around it.
    This creates volume and pushes price towards a random walk path.
    """
    def __init__(self, agent_id, cash, inventory, arrival_rate=0.5, volatility=0.02):
        super().__init__(agent_id, cash, inventory)
        self.arrival_rate = arrival_rate
        self.volatility = volatility # Volatility of their price estimate

    def get_action(self, market_snapshot, fair_value):
        """
        Requires 'fair_value' to determine order placement.
        """
        # 1. Poisson Arrival Check
        if np.random.random() > self.arrival_rate:
            return None

        # 2. Determine Side (Random 50/50)
        side = 'buy' if np.random.random() > 0.5 else 'sell'
        
        # 3. Determine Price (Fair Value + Agent Noise)
        # Even if the fair value is 100, this agent might think it's 101 or 99
        noise = np.random.normal(0, self.volatility * fair_value)
        order_price = round(fair_value + noise, 2)
        
        # 4. Determine Quantity (Random small lots)
        qty = np.random.randint(1, 20)
        
        # 5. Sanity Check (Budget)
        if side == 'buy' and self.cash < order_price * qty:
            return None
        # We ignore inventory checks for noise traders to keep volume high
        
        return {
            'agent_id': self.agent_id,
            'side': side,
            'price': order_price,
            'qty': qty,
            'type': 'limit' # Noise traders use Limit orders to "rest" in the book
        }